In [1]:
import numpy as np
import scipy.stats as stats
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
PUBLIC_SRC = 'https://seshat.datasd.org/get_it_done_reports'
YEARS = [2021, 2022, 2023, 2024, 2025, 2026]
PRIVATE_CSV = 'data/gid_cases-001.csv'
REFFERALS_CSV = 'data/referral data for UCSD-2026-11-02-23-09-16.csv'
ATTACHMENTS_CSV = 'data/public-attachments-cases.csv'
CLAIMS_CSV = 'data/claims_claims.csv'
SURVEYS_CSV = 'data/gid_case_closure_survey.csv'
DATE_START, DATE_END = '2023-01-01', '2026-10-01'

# loading get it done cases data

## cases information

In [3]:
PUB_COLS = [
    'service_request_id', 'service_request_parent_id', 'date_requested', 'case_age_days', 'case_record_type', 'service_name',
    'service_name_detail', 'date_closed', 'status', 'case_origin', 'referred', 'public_description'
    ]

closed = pd.concat([pd.read_csv(f'{PUBLIC_SRC}/get_it_done_requests_closed_{y}_datasd.csv', usecols=PUB_COLS) for y in YEARS], ignore_index=True)
open_ = pd.read_csv(f'{PUBLIC_SRC}/get_it_done_requests_open_datasd.csv', usecols=PUB_COLS)

In [4]:
pub = pd.concat([closed, open_], ignore_index=True).drop_duplicates('service_request_id')

pub['date_requested'] = pd.to_datetime(pub['date_requested'])
pub['date_requested'] = pub['date_requested'].dt.tz_localize('UTC').dt.tz_convert('US/Pacific')
pub['date_closed'] = pd.to_datetime(pub['date_closed'])

pub = pub[(pub['date_requested'] >= DATE_START) & (pub['date_requested'] < DATE_END)].copy()

In [5]:
PRIV_COLS = [
    'After Photo in Salesforce or AWS', 'Case Reason', 'Last Modified Date', 'Parent Case ID', 'Priority',
    'Parent Case ID', 'Case Number', 'Owner ID', 'Last Modified By ID', 'Created By ID', 'Created Date'
    ]

RENAME = {t:t.lower().replace(' ', '_') for t in PRIV_COLS}
RENAME.update({'Status': 'private_status'})
priv = pd.read_csv(PRIVATE_CSV, usecols=list(RENAME), dtype=str).rename(columns=RENAME)

In [6]:
priv['case_number'] = pd.to_numeric(priv['case_number'], errors='coerce')
priv = priv.dropna(subset=['case_number'])
priv['case_number'] = priv['case_number'].astype(int)
priv.drop_duplicates('case_number', inplace=True)

priv['last_modified_date'] = pd.to_datetime(priv['last_modified_date'])
priv['case_number'] = priv['case_number'].astype(float)
# priv.drop(columns=['owner_id', 'last_modified_by_id'], inplace=True)

In [7]:
cases_data = pub.merge(priv, left_on='service_request_id', right_on='case_number', how='left')

In [8]:
cases_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1569710 entries, 0 to 1569709
Data columns (total 23 columns):
 #   Column                            Non-Null Count    Dtype                     
---  ------                            --------------    -----                     
 0   service_request_id                1569710 non-null  int64                     
 1   service_request_parent_id         212380 non-null   float64                   
 2   date_requested                    1569710 non-null  datetime64[ns, US/Pacific]
 3   case_age_days                     1569710 non-null  float64                   
 4   case_record_type                  1569710 non-null  object                    
 5   service_name                      1565773 non-null  object                    
 6   service_name_detail               923202 non-null   object                    
 7   date_closed                       1509435 non-null  datetime64[ns]            
 8   status                            1569710 

In [9]:
# Inplace function(s) and code to clean up referred column. Based on the frequency of referrals made
def categorize_referrals(txt):
    txt = str(txt).lower()
    if 'sdge' in txt or 'sdg&e' in txt:
        return 'sdg&e'
    if 'cox.com' in txt:
        return 'cox communications'
    if 'charter.com' in txt or 'spectrum' in txt:
        return 'spectrum'
    if 'att' in txt or 'at&t' in txt:
        return 'at&t'
    if 'cleanharbors' in txt:
        return 'clean harbors sanitation'
    if 'wm.com' in txt or 'waste management' in txt:
        return 'waste management'
    if 'republicservices' in txt or 'republic-services' in txt:
        return 'republic services'
    if 'edcodisposal' in txt:
        return 'edco disposal'
    if 'sdmts' in txt:
        return 'mts'
    if 'nctd.org' in txt:
        return 'nctd'
    if 'bnsf' in txt:
        return 'bnsf railroad'
    if 'caltrans' in txt or 'dot.ca.gov' in txt:
        return 'caltrans'
    if 'portofsandiego' in txt or 'port of san diego' in txt:
        return 'port of sd'
    if 'san.org' in txt:
        return 'sd international airport'
    if 'sandi.net' in txt:
        return 'sd unified school district'
    if 'ucsd.edu' in txt:
        return 'ucsd'
    if 'sdsu.edu' in txt:
        return 'sdsu'
    if 'sdccd.edu' in txt:
        return 'sdccd'
    if 'chulavistaca.gov' in txt:
        return 'city of chula vista'
    if 'sdcounty.ca.gov' in txt:
        return 'san diego county'
    if 'parks.ca.gov' in txt or 'wildlife.ca.gov' in txt:
        return 'california state parks/wildlife'
    if 'usps' in txt:
        return 'usps'
    if 'livelmh' in txt:
        return 'liberty military housing'
    if 'improvedtsd' in txt or 'ballparkdistrict' in txt or 'downtownsandiego.org' in txt:
        return 'downtown clean & safe / ballpark'
    if 'sdhumane' in txt:
        return 'sd humane society'
    if 'vector' in txt:
        return 'county vector control'
    if 'rms' in txt or 'retailsolutions' in txt or 'retailms.net' in txt:
        return 'retail solutions (rms)'
    if 'smartcart' in txt:
        return 'smart cart'
    if 'clearchannel' in txt:
        return 'clear channel'
    if 'outfront.com' in txt:
        return 'outfront media'
    if 'engineering' in txt:
        return 'field engineering'
    if 'facilitiesmaint' in txt:
        return 'facilities maintenance'
    if 'treepermit' in txt:
        return 'tree permits'
    if 'publicliability' in txt or 'riskmanagement' in txt:
        return 'risk management / public liability'
    if 'specialevents' in txt:
        return 'special events'
    if 'realestate' in txt:
        return 'real estate assets'
    if 'sdpd' in txt or 'police' in txt:
        return 'sdpd'
    if 'sdfd' in txt or 'fire' in txt:
        return 'sdfd'
    if 'wcm_gid' in txt or 'water' in txt:
        return 'water / wastewater (pud)'
    if 'jkafulides' in txt or 'parking' in txt:
        return 'parking meter repair'
    if 'sdlg' in txt or 'lifeguard' in txt:
        return 'lifeguards'
    if 'fit_dsd' in txt or 'development services' in txt:
        return 'development services (dsd)'
    if 'sdbusiness' in txt:
        return 'sd business / economic dev'
    if 'street division' in txt:
        return 'street division'
    if 't-row code compliance' in txt:
        return 't-row code compliance'     
    if 'sanitization contractor' in txt or 'environmental services' in txt or 'esd' in txt:
        return 'environmental services (esd)'
    if 'parks intake' in txt:
        return 'parks & recreation'
    if 'testing only' in txt:
        return 'test/invalid'
    if 'null' in txt:
        return 'error'
    return txt

def clean_referrals(df):
    df['referred'] = df['referred'].apply(categorize_referrals)
    # If the referral is not a specific organization (is an email), extract the email:
    df.loc[df['referred'].str.contains('@'), 'referred'] = df.loc[df['referred'].str.contains('@'), 'referred'].str.extract(r'([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})')[0].str.lower()
    df.loc[df['referred'] == 'nan', 'referred'] = np.nan

In [10]:
clean_referrals(cases_data)

## referrals

In [11]:
REF_COLS = ['Internal Case Referral Flag', 'Internal Referral Count']

REF_RENAME = {t:t.lower().replace(' ', '_') for t in REF_COLS}
REF_RENAME.update({'Case Number': 'ref_case_number', 'Status': 'referral_status'})
referrals_info = pd.read_csv(REFFERALS_CSV, encoding='latin-1', usecols=list(REF_RENAME)).rename(columns=REF_RENAME)

In [12]:
df = cases_data.merge(referrals_info, left_on='service_request_id', right_on='ref_case_number', how='left').drop(columns='ref_case_number')

In [13]:
df = df[df['status'].isin(['Closed', 'Referred'])]

In [14]:
df['last_modified_date'] = df['last_modified_date'].dt.tz_localize('US/Pacific', ambiguous=True)

In [15]:
df.head()

,service_request_id,service_request_parent_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,case_origin,...,last_modified_by_id,last_modified_date,owner_id,parent_case_id,priority,case_reason,private_status,internal_case_referral_flag,internal_referral_count,referral_status
0,4470480,NaN,2023-10-20 01:24:00-07:00,0.0,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2023-10-20,Closed,Worker App,...,005t0000003YOfBAAW,2023-10-20 08:24:53-07:00,00Gt0000000TfYFEA0,NaN,Medium,Problem resolved,Closed,NaN,NaN,NaN
1,4470495,NaN,2023-10-20 01:28:00-07:00,0.0,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2023-10-20,Closed,Worker App,...,005t0000003YOfBAAW,2023-10-20 08:28:43-07:00,00Gt0000000TfYFEA0,NaN,Medium,Problem resolved,Closed,NaN,NaN,NaN
2,4470497,NaN,2023-10-20 01:28:00-07:00,3.0,Parking,Parking Violation,Red Zone,2023-10-23,Closed,Mobile,...,0058z000000aVjkAAE,2023-10-23 12:52:12-07:00,00Gt0000000TfZ7EAK,NaN,Medium,NaN,Closed,0.0,0.0,Closed
3,4470499,4470478.0,2023-10-20 01:28:00-07:00,6.0,Storm Water Code Enforcement,Stormwater Code Enforcement,SWPP - Storm Water Polution Prevention,2023-10-26,Closed,Mobile,...,005t0000002tdfuAAA,2023-12-19 05:39:15-08:00,005t00000025QxyAAE,5008z000006VfAxAAK,Medium,Not a Storm Water Issue,Closed,0.0,0.0,Closed
4,4470521,NaN,2023-10-20 01:33:00-07:00,0.0,TSW,Graffiti - Public,NaN,2023-10-20,Closed,Crew/Self Generated,...,0058z000000efccAAA,2025-03-18 12:57:47-07:00,00G8z000000Nk02EAC,NaN,Low,Problem resolved,Closed,NaN,NaN,NaN


## attachments

In [16]:
PHOTOS_RENAME = {'Id': 'photo_location', 'Parent.Name': 'photo_case_number'}
photos_data = pd.read_csv(ATTACHMENTS_CSV, usecols=list(PHOTOS_RENAME)).rename(columns=PHOTOS_RENAME)

In [17]:
photos_data.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2588353 entries, 0 to 2588352
Data columns (total 2 columns):
 #   Column             Non-Null Count    Dtype 
---  ------             --------------    ----- 
 0   photo_location     2588353 non-null  object
 1   photo_case_number  2588353 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 39.5+ MB


# data sanity check

### `case_age_days` < 0

In [18]:
df.shape

(1509435, 26)

In [19]:
df[df.duplicated(subset='service_request_id', keep=0)].shape

(0, 26)

In [20]:
df.shape

(1509435, 26)

In [21]:
print(df.loc[df['case_age_days'] < 0., 'service_request_parent_id'].nunique())
df.loc[df['case_age_days'] < 0., ['service_request_id', 'service_request_parent_id', 'date_requested', 'case_age_days', 'case_record_type', 'service_name', 'service_name_detail', 'date_closed', 'created_by_id', 'last_modified_by_id', 'last_modified_date', 'owner_id', 'case_reason', 'internal_case_referral_flag', 'internal_referral_count', 'referral_status']].sort_values(by='service_request_parent_id')

8


,service_request_id,service_request_parent_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,created_by_id,last_modified_by_id,last_modified_date,owner_id,case_reason,internal_case_referral_flag,internal_referral_count,referral_status
1188455,5478324,5184714.0,2025-12-05 02:17:00-08:00,-15.0,TSW,Traffic Sign Maintenance,NaN,2025-11-20,005t0000000F4uMAAS,005t0000000m7xPAAQ,2026-03-02 12:15:39-08:00,005t0000003A2RWAA0,No Email,0.0,0.0,Closed
1188555,5505976,5239649.0,2025-12-27 12:21:00-08:00,-131.0,Parks & Recreation,Parks Issue,Other,2025-08-18,005t0000000F4uMAAS,005t0000000m7xPAAQ,2026-09-03 14:14:10-07:00,005t0000000F9BMAA0,No Email,1.0,1.0,Closed
1188628,5564848,5239649.0,2026-01-31 00:54:00-08:00,-166.0,Parks & Recreation,Parks Issue,Other,2025-08-18,005t0000000F4uMAAS,005t0000000m7xPAAQ,2026-09-03 14:14:10-07:00,005t0000000F9BMAA0,No Email,1.0,1.0,Closed
1188520,5443648,5404526.0,2025-11-08 01:10:00-08:00,-22.0,Parks & Recreation,Tree Maintenance,EVALUATE TREE FOR REMOVAL,2025-10-17,005t0000000F4uMAAS,005t0000001kdcLAAQ,2026-01-14 09:59:20-08:00,00G8z000000MruOEAS,Issue Resolved,0.0,0.0,Closed
1188621,5508054,5419383.0,2025-12-29 09:16:00-08:00,-41.0,TSW ROW,Vegetation Encroachment,NaN,2025-11-18,005t0000000F8MTAA0,005t0000001kdcLAAQ,2026-01-08 10:59:15-08:00,00G8z000000XBhmEAG,Compliance Achieved,1.0,1.0,Closed
1188565,5606526,5426964.0,2026-03-01 03:14:00-08:00,-124.0,Parks & Recreation,Parks Issue,Potential Tree Hazard,2025-10-28,005t0000000F4uMAAS,005t0000001kdcLAAQ,2026-04-14 19:55:07-07:00,00G8z000000MruPEAS,No Email,1.0,1.0,Closed
1218435,5517031,5510796.0,2026-01-05 13:24:00-08:00,-3.0,TSW,Stormwater,DRAIN INLET,2026-01-02,005t0000000F4uMAAS,005t0000000m7xPAAQ,2026-03-02 12:11:46-08:00,005cs00000KmZJJAA3,No Email,0.0,0.0,Closed
1327684,5617540,5611411.0,2026-03-08 06:58:00-07:00,-3.0,Parks & Recreation,Parks Issue,Other,2026-03-05,005t0000000F8MTAA0,005t0000000m7xPAAQ,2026-09-03 14:14:10-07:00,005cs00000L4GV9AAN,No Email,1.0,1.0,Closed
1221896,5783579,5756076.0,2026-06-29 04:14:00-07:00,-7.0,Parks & Recreation,Parks Issue,Potential Tree Hazard,2026-06-22,005t0000000F4uMAAS,005t0000000m7xPAAQ,2026-09-03 14:11:43-07:00,005t0000003A2RWAA0,No Email,1.0,1.0,Closed


Each case with a negative `case_age_day` has a non empty `service_request_parent_id`. Additionally, the `created_by_id`s and `last_modified_by_id`s are nonunique.

In [22]:
parents = df.loc[df['case_age_days'] < 0., 'service_request_parent_id'].values
df.loc[df['service_request_id'].isin(parents), ['service_request_id', 'service_request_parent_id', 'date_requested', 'case_age_days', 'case_record_type', 'service_name', 'service_name_detail', 'date_closed', 'created_by_id', 'last_modified_by_id', 'last_modified_date', 'owner_id', 'case_reason', 'internal_case_referral_flag', 'internal_referral_count', 'referral_status']].sort_values(by='service_request_id')

,service_request_id,service_request_parent_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,created_by_id,last_modified_by_id,last_modified_date,owner_id,case_reason,internal_case_referral_flag,internal_referral_count,referral_status
1188513,5184714,NaN,2025-04-14 23:27:00-07:00,219.0,TSW,Traffic Sign Maintenance,NaN,2025-11-20,005t0000000F4uMAAS,005t0000000m7xPAAQ,2026-03-02 12:15:39-08:00,00G61000001Bq9xEAC,No Email,0.0,0.0,Closed
1188335,5239649,NaN,2025-06-01 08:12:00-07:00,78.0,TSW,Tree Maintenance,INVESTIGATE,2025-08-18,005t0000000F4uMAAS,005t0000000m7xPAAQ,2026-09-03 14:14:10-07:00,005t0000000F9BMAA0,No Email,1.0,6.0,Closed
1188351,5404526,NaN,2025-10-08 07:47:00-07:00,9.0,Parks & Recreation,Parks Issue,Potential Tree Hazard,2025-10-17,005t0000000F4uMAAS,0058z000000vQNLAA2,2026-01-14 09:54:01-08:00,00G8z000000MruOEAS,Issue Resolved,1.0,1.0,Closed
1188360,5419383,NaN,2025-10-21 08:48:00-07:00,28.0,TSW ROW,Vegetation Encroachment,NaN,2025-11-18,005t0000000F8MTAA0,005t0000001kdcLAAQ,2026-01-09 00:04:02-08:00,00G8z000000XBhmEAG,Compliance Achieved,1.0,1.0,Closed
1188368,5426964,NaN,2025-10-27 08:28:00-07:00,1.0,Parks & Recreation,Parks Issue,Potential Tree Hazard,2025-10-28,005t0000000F4uMAAS,005t0000005uWGAAA2,2026-04-14 19:34:03-07:00,00G8z000000MruPEAS,No Email,1.0,1.0,Closed
1240594,5510796,NaN,2026-01-01 00:14:00-08:00,1.0,TSW,Stormwater,DRAIN INLET,2026-01-02,005t0000000F4uMAAS,005t0000000m7xPAAQ,2026-03-02 12:11:52-08:00,005cs00000KmZJJAA3,No Email,0.0,0.0,Closed
1276860,5611411,NaN,2026-03-04 02:10:00-08:00,1.0,TSW,Tree Maintenance,INVESTIGATE,2026-03-05,005t0000000F8MTAA0,005t0000000m7xPAAQ,2026-09-03 14:14:10-07:00,005cs00000L4GV9AAN,No Email,1.0,2.0,Closed
1404103,5756076,NaN,2026-06-09 08:15:00-07:00,13.0,TSW,Tree Maintenance,EVALUATE TREE FOR REMOVAL,2026-06-22,005t0000000F4uMAAS,005t0000000m7xPAAQ,2026-09-03 14:11:48-07:00,005t0000003A2RWAA0,No Email,1.0,2.0,Closed


Intuitively, the cases with negative ages seem to be an attempt at correcting the parent case or a return to intake that creates a new case. Notably, the city group (`case_record_type`) and service details differ between each case and its parent.

Even so, this looks like an administrative error. Because we lack the necessary information to resolve this issue, we drop the cases and their parents.

In [23]:
df = df[~((df['service_request_parent_id'].isin(parents)) | (df['service_request_id'].isin(parents)))]

### `case_age_days` == 0

In [24]:
zero_days_resolution = df.loc[df['case_age_days'] == 0, ['service_request_id', 'date_requested', 'case_record_type', 'service_name', 'service_name_detail', 'date_closed', 'public_description', 'status', 'referred', 'case_origin', 'case_reason', 'priority', 'internal_case_referral_flag', 'internal_referral_count', 'after_photo_in_salesforce_or_aws']]
zero_days_resolution

,service_request_id,date_requested,case_record_type,service_name,service_name_detail,date_closed,public_description,status,referred,case_origin,case_reason,priority,internal_case_referral_flag,internal_referral_count,after_photo_in_salesforce_or_aws
0,4470480,2023-10-20 01:24:00-07:00,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2023-10-20,NaN,Closed,NaN,Worker App,Problem resolved,Medium,NaN,NaN,FALSE
1,4470495,2023-10-20 01:28:00-07:00,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2023-10-20,NaN,Closed,NaN,Worker App,Problem resolved,Medium,NaN,NaN,FALSE
4,4470521,2023-10-20 01:33:00-07:00,TSW,Graffiti - Public,NaN,2023-10-20,NaN,Closed,NaN,Crew/Self Generated,Problem resolved,Low,NaN,NaN,TRUE
5,4470529,2023-10-20 01:35:00-07:00,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2023-10-20,NaN,Closed,NaN,Worker App,Problem resolved,Medium,NaN,NaN,FALSE
7,4470540,2023-10-20 01:38:00-07:00,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2023-10-20,NaN,Closed,NaN,Worker App,Problem resolved,Medium,NaN,NaN,FALSE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1509405,6130493,2026-09-30 13:47:47-07:00,TSW,Other,NaN,2026-09-30,Looks like a stolen car was dumped in front of...,Referred,sdpd,Mobile,Externally Referred,Medium,0.0,0.0,FALSE
1509413,6130560,2026-09-30 22:21:07-07:00,TSW,Pothole,NaN,2026-10-01,Sinkhole forming quickly!!! \nI sent in a repo...,Closed,NaN,Mobile,NaN,Medium,0.0,0.0,TRUE
1509416,6130588,2026-09-30 23:49:23-07:00,Parking,Parking Violation,Lane of Traffic,2026-10-01,Flat bed truck parked for an hour in the east ...,Closed,NaN,Mobile,GOA,Medium,0.0,0.0,TRUE
1509417,6130595,2026-09-30 23:55:19-07:00,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2026-10-01,NaN,Closed,NaN,Worker App,Problem resolved,Medium,NaN,NaN,FALSE


In [25]:
zero_days_resolution

,service_request_id,date_requested,case_record_type,service_name,service_name_detail,date_closed,public_description,status,referred,case_origin,case_reason,priority,internal_case_referral_flag,internal_referral_count,after_photo_in_salesforce_or_aws
0,4470480,2023-10-20 01:24:00-07:00,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2023-10-20,NaN,Closed,NaN,Worker App,Problem resolved,Medium,NaN,NaN,FALSE
1,4470495,2023-10-20 01:28:00-07:00,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2023-10-20,NaN,Closed,NaN,Worker App,Problem resolved,Medium,NaN,NaN,FALSE
4,4470521,2023-10-20 01:33:00-07:00,TSW,Graffiti - Public,NaN,2023-10-20,NaN,Closed,NaN,Crew/Self Generated,Problem resolved,Low,NaN,NaN,TRUE
5,4470529,2023-10-20 01:35:00-07:00,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2023-10-20,NaN,Closed,NaN,Worker App,Problem resolved,Medium,NaN,NaN,FALSE
7,4470540,2023-10-20 01:38:00-07:00,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2023-10-20,NaN,Closed,NaN,Worker App,Problem resolved,Medium,NaN,NaN,FALSE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1509405,6130493,2026-09-30 13:47:47-07:00,TSW,Other,NaN,2026-09-30,Looks like a stolen car was dumped in front of...,Referred,sdpd,Mobile,Externally Referred,Medium,0.0,0.0,FALSE
1509413,6130560,2026-09-30 22:21:07-07:00,TSW,Pothole,NaN,2026-10-01,Sinkhole forming quickly!!! \nI sent in a repo...,Closed,NaN,Mobile,NaN,Medium,0.0,0.0,TRUE
1509416,6130588,2026-09-30 23:49:23-07:00,Parking,Parking Violation,Lane of Traffic,2026-10-01,Flat bed truck parked for an hour in the east ...,Closed,NaN,Mobile,GOA,Medium,0.0,0.0,TRUE
1509417,6130595,2026-09-30 23:55:19-07:00,ESD Complaint/Report,Illegal Dumping,Illegal Dumping,2026-10-01,NaN,Closed,NaN,Worker App,Problem resolved,Medium,NaN,NaN,FALSE


In [26]:
zero_days_resolution['case_record_type'].value_counts()

case_record_type
ESD Complaint/Report            245261
TSW                             131819
Parking                          41581
Neighborhood Policing             7555
Parks & Recreation                3589
DSD                               1482
Storm Water Code Enforcement       916
TSW ROW                            729
Traffic Engineering                175
Special Situation                    1
Name: count, dtype: int64

In [27]:
zero_days_resolution['referred'].value_counts(dropna=False)

referred
NaN                         377181
sdpd                         10878
water / wastewater (pud)      9230
sdg&e                         6574
caltrans                      6224
                             ...  
marasheed@sandiego.gov           1
arts@sandiego.gov                1
snaghavian@sandiego.gov          1
knichols@sandiego.gov            1
vlubuguin@sandiego.gov           1
Name: count, Length: 417, dtype: int64

In [28]:
zero_days_resolution['case_origin'].value_counts()

case_origin
Worker App             191401
Mobile                 118679
Crew/Self Generated     46543
Web                     44764
Phone                   21726
CC Self Generate         8594
GID Field                 702
Construction Patrol       240
Self-Generated            127
Hauler List                99
Email                      81
Residential Patrol         59
Famosa Patrol              26
System Generated           10
Irrigation Patrol           9
Internal                    4
Referral Notice             2
Council                     2
FO Self Generate            2
Claim Related               1
Name: count, dtype: int64

Some cases are directly routed with no intake review. We see this especially with `Parking Violations` requests and cases sent to `sdpd`. Basically, if the case should evidently be referred elsewhere, it will automatically be referred elsewhere. Many of these 0 days cases are also generated by workers, implying that they submitted a request then promptly dealt with it onsight.

In [29]:
photos_data

,photo_location,photo_case_number
0,00P8z000000JwgbEAC,3607671
1,00P8z000000JwhfEAC,3607677
2,00P8z000000JwheEAC,3607677
3,00P8z000000Jwm4EAC,3607693
4,00P8z000000Jwm3EAC,3607693
...,...,...
2588348,00Pt000000bEuIREA0,3607640
2588349,00Pt000000bEuJ5EAK,3607644
2588350,00Pt000000bEuIOEA0,3607664
2588351,00Pt000000bEuINEA0,3607664


In [30]:
photo_examples = zero_days_resolution.merge(photos_data, left_on='service_request_id', right_on='photo_case_number').drop(columns='photo_case_number')

In [31]:
url = lambda t: f'https://getitdone.sandiego.gov/servlet/servlet.FileDownload?file={t}'

In [32]:
photo_examples['case_record_type'].value_counts()

case_record_type
TSW                             85995
Parking                         61007
ESD Complaint/Report            22872
Neighborhood Policing            8354
DSD                              1418
Parks & Recreation               1183
TSW ROW                           715
Storm Water Code Enforcement      512
Traffic Engineering               176
Name: count, dtype: int64

In [33]:
random_photo_example = photo_examples.sample(1)
random_photo_example

,service_request_id,date_requested,case_record_type,service_name,service_name_detail,date_closed,public_description,status,referred,case_origin,case_reason,priority,internal_case_referral_flag,internal_referral_count,after_photo_in_salesforce_or_aws,photo_location
98415,5183665,2025-04-14 02:25:00-07:00,TSW,Other,NaN,2025-04-14,Running water,Referred,water / wastewater (pud),Mobile,Externally Referred,Medium,0.0,0.0,FALSE,00Pcs00000eNKS7EAO


In [34]:
print(random_photo_example['public_description'].values[0])
url(random_photo_example['photo_location'].values[0])

Running water


'https://getitdone.sandiego.gov/servlet/servlet.FileDownload?file=00Pcs00000eNKS7EAO'

In [35]:
sdpd_photo_example = photo_examples[(photo_examples['case_record_type'] == 'TSW') & (photo_examples['referred'] == 'sdpd')].sample(1)
sdpd_photo_example
# 4172624

,service_request_id,date_requested,case_record_type,service_name,service_name_detail,date_closed,public_description,status,referred,case_origin,case_reason,priority,internal_case_referral_flag,internal_referral_count,after_photo_in_salesforce_or_aws,photo_location
83430,4999761,2024-11-08 23:07:00-08:00,TSW,ROW Maintenance,TRAFFIC DEVICES,2024-11-09,The apartment at 4365-1/2 Oregon street may me...,Referred,sdpd,Mobile,Externally Referred,Medium,0.0,0.0,FALSE,00Pcs00000IVIazEAH


In [36]:
print(sdpd_photo_example['public_description'].values[0])
url(sdpd_photo_example['photo_location'].values[0])

The apartment at 4365-1/2 Oregon street may me selling drugs. Vehicles are parking for short periods throughout the night


'https://getitdone.sandiego.gov/servlet/servlet.FileDownload?file=00Pcs00000IVIazEAH'

In [37]:
photo_examples['case_origin'].value_counts()

case_origin
Mobile              138878
Web                  43162
Worker App             158
Phone                   29
CC Self Generate         5
Name: count, dtype: int64

In [38]:
worker_photo_examples = photo_examples[(photo_examples['case_origin'] == 'Worker App')].sample(10)
worker_photo_examples
# url(worker_photo_example['photo_location'].values[0])

,service_request_id,date_requested,case_record_type,service_name,service_name_detail,date_closed,public_description,status,referred,case_origin,case_reason,priority,internal_case_referral_flag,internal_referral_count,after_photo_in_salesforce_or_aws,photo_location
11566,4366768,2023-08-10 01:01:00-07:00,ESD Complaint/Report,Biohazard,NaN,2023-08-10,NaN,Referred,clean harbors sanitation,Worker App,Externally Referred,Medium,NaN,NaN,FALSE,00P8z0000034fGuEAI
103827,5448453,2025-11-12 02:10:00-08:00,ESD Complaint/Report,Environmental Services Code Compliance,Hazardous Waste,2025-11-12,NaN,Referred,clean harbors sanitation,Worker App,Externally Referred,Medium,NaN,NaN,FALSE,00Pcs000019v5V7EAI
5751,4350131,2023-07-31 03:04:00-07:00,ESD Complaint/Report,Biohazard,NaN,2023-07-31,NaN,Referred,clean harbors sanitation,Worker App,Externally Referred,Medium,NaN,NaN,FALSE,00P8z0000030lVSEAY
49113,4755831,2024-05-16 05:47:00-07:00,ESD Complaint/Report,Biohazard,NaN,2024-05-16,NaN,Referred,clean harbors sanitation,Worker App,Externally Referred,Medium,NaN,NaN,FALSE,00P8z00000492ZgEAI
737,4310233,2023-07-05 02:45:00-07:00,ESD Complaint/Report,Biohazard,NaN,2023-07-05,NaN,Referred,clean harbors sanitation,Worker App,Externally Referred,Medium,NaN,NaN,FALSE,00P8z000002uh80EAA
75404,4953772,2024-10-03 01:04:00-07:00,ESD Complaint/Report,NaN,NaN,2024-10-03,NaN,Referred,clean harbors sanitation,Worker App,Externally Referred,Medium,NaN,NaN,FALSE,00Pcs00000CG2oZEAT
97702,5185184,2025-04-15 03:42:00-07:00,ESD Complaint/Report,NaN,NaN,2025-04-15,NaN,Referred,clean harbors sanitation,Worker App,Externally Referred,Medium,NaN,NaN,FALSE,00Pcs00000elMJ1EAM
140626,5939468,2026-08-13 00:43:00-07:00,ESD Complaint/Report,Biohazard,NaN,2026-08-13,NaN,Referred,clean harbors sanitation,Worker App,Externally Referred,Medium,NaN,NaN,FALSE,00Pcs00001s4RWhEAM
24622,4304014,2023-06-30 01:10:00-07:00,ESD Complaint/Report,Biohazard,NaN,2023-06-30,NaN,Referred,clean harbors sanitation,Worker App,Externally Referred,Medium,NaN,NaN,FALSE,00P8z000002uJbMEAU
62490,4887756,2024-08-16 00:50:00-07:00,ESD Complaint/Report,Biohazard,NaN,2024-08-16,NaN,Referred,clean harbors sanitation,Worker App,Externally Referred,Medium,NaN,NaN,FALSE,00Pcs000004QoQJEA0


`case_origin` `'Worker App'` are made specifically by clean harbors sanitation. It appears they submit the service then perform their **biohazard** work immediately. Not all of these images are meaningful.

In [39]:
df.loc[df['service_name'] == 'Biohazard', 'case_age_days'].describe()

count    483.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: case_age_days, dtype: float64

In [40]:
url(worker_photo_examples['photo_location'].sample(1).values[0])

'https://getitdone.sandiego.gov/servlet/servlet.FileDownload?file=00P8z000002uJbMEAU'

In [41]:
parking_photo_examples = photo_examples[(photo_examples['case_record_type'] == 'Parking')].sample(10)
parking_photo_examples
# url(worker_photo_example['photo_location'].values[0])

,service_request_id,date_requested,case_record_type,service_name,service_name_detail,date_closed,public_description,status,referred,case_origin,case_reason,priority,internal_case_referral_flag,internal_referral_count,after_photo_in_salesforce_or_aws,photo_location
128527,5249340,2025-06-09 04:29:00-07:00,Parking,Parking Violation,Red Zone,2025-06-09,The truck is blocking the driveway exit and pa...,Closed,NaN,Mobile,GOA,Medium,0.0,0.0,TRUE,00Pcs00000mbVvdEAE
158991,5787152,2026-07-01 09:20:00-07:00,Parking,Parking Violation,Blocked Driveway,2026-07-01,Blocking driveway,Closed,NaN,Mobile,Cited,Medium,0.0,0.0,TRUE,00Pcs00001k3tJYEAY
172132,5775935,2026-06-23 22:56:00-07:00,Parking,Parking Violation,Fire Hydrant,2026-06-24,This vehicle is parked in front of the fire hy...,Closed,NaN,Mobile,Other (+ mandatory comments),Medium,0.0,0.0,FALSE,00Pcs00001inhMvEAI
83887,5005415,2024-11-14 01:53:00-08:00,Parking,Parking Violation,Sidewalk,2024-11-14,Parking on sidewalk,Closed,NaN,Mobile,Moved,Medium,0.0,0.0,TRUE,00Pcs00000JGZweEAH
23563,4159109,2023-03-23 03:46:00-07:00,Parking,Parking,Passenger Zone (White),2023-03-23,n:Parking in a passenger loading zone during s...,Closed,NaN,Web,NaN,Medium,0.0,0.0,FALSE,00P8z000002a8qHEAQ
174407,5813072,2026-07-17 08:58:00-07:00,Parking,Parking Violation,Parking Within 20 Feet of Crosswalk,2026-07-17,"Since the closing of Avenida de La Playa, ther...",Closed,NaN,Web,GOA,Medium,0.0,0.0,TRUE,00Pcs00001mun8NEAQ
161402,5715308,2026-05-12 07:43:00-07:00,Parking,Parking Violation,Timed Zone Violation,2026-05-12,This car Parks all day in a 30-minute slot. It...,Closed,NaN,Mobile,Cited,Medium,0.0,0.0,TRUE,00Pcs00001bqKODEA2
164433,5755900,2026-06-09 06:47:00-07:00,Parking,Parking Violation,Parking Zone Violation,2026-06-09,Vehicles parked blocking sidewalk access. ADA ...,Closed,NaN,Mobile,Other (+ mandatory comments),Medium,1.0,1.0,FALSE,00Pcs00001gOz9kEAC
108487,5199969,2025-04-28 03:09:00-07:00,Parking,Parking Violation,Passenger Zone (White),2025-04-28,This Toyota truck has been in the loading zone...,Closed,NaN,Mobile,GOA,Medium,0.0,0.0,TRUE,00Pcs00000gPLPLEA4
72417,4943058,2024-09-24 23:42:00-07:00,Parking,Parking Violation,Sidewalk,2024-09-25,"Pick up truck blocking sidewalk, this is a dai...",Closed,NaN,Mobile,NaN,Medium,0.0,0.0,FALSE,00Pcs00000A83seEAB


In [42]:
url(parking_photo_examples['photo_location'].sample(1).values[0])

'https://getitdone.sandiego.gov/servlet/servlet.FileDownload?file=00Pcs00000JGZweEAH'

In [43]:
# code for bar chart

## other data sanity checks

### extremely high `case_age_days`

In [44]:
df['case_age_days'].describe().astype(int)

count    1509417
mean          25
std           85
min            0
25%            0
50%            3
75%           13
max         1400
Name: case_age_days, dtype: int64

In [45]:
df['date_closed'] = pd.to_datetime(df['date_closed']).dt.tz_localize('US/Pacific')
outliers = df[df['case_age_days'] >= 365].sort_values(by='case_age_days', ascending=False)
outliers

,service_request_id,service_request_parent_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,case_origin,...,last_modified_by_id,last_modified_date,owner_id,parent_case_id,priority,case_reason,private_status,internal_case_referral_flag,internal_referral_count,referral_status
1293564,4050591,4012642.0,2023-01-05 06:36:12-08:00,1400.0,TSW,Street Light Maintenance,STREET LIGHT OUT,2026-11-05 00:00:00-08:00,Closed,Web,...,005cs00000cYlmbAAC,2025-10-16 22:28:15-07:00,005t0000000F9BMAA0,5008z000004KN3iAAG,Medium,Problem resolved,Duplicate,0.0,0.0,Duplicate
1293551,4045868,NaN,2023-01-02 02:51:22-08:00,1396.0,ESD Complaint/Report,Trash/Recycling Collection,Damaged Container,2026-10-29 00:00:00-07:00,Closed,Mobile,...,005cs00000cYlmbAAC,2025-10-16 22:28:48-07:00,00Gt0000000TfYkEAK,NaN,Medium,Problem resolved,New,1.0,1.0,Closed
1293604,4058245,NaN,2023-01-10 14:51:18-08:00,1395.0,TSW,ROW Maintenance,MARKERS/DELINIATORS - MAINTENANCE,2026-11-05 00:00:00-08:00,Closed,Mobile,...,005cs00000cYlmbAAC,2025-10-16 22:26:47-07:00,00G61000001Bq9xEAC,NaN,Medium,Problem resolved,Acknowledged,0.0,0.0,Acknowledged
1293559,4048515,NaN,2023-01-04 02:21:04-08:00,1392.0,TSW,Traffic Sign Maintenance,NaN,2026-10-27 00:00:00-07:00,Closed,Web,...,0058z000000ojlxAAA,2026-10-23 11:57:06-07:00,005t0000009WRrSAAW,NaN,Medium,Problem resolved,Acknowledged,0.0,0.0,Closed
1293585,4056554,3972835.0,2023-01-09 13:02:15-08:00,1390.0,TSW,Street Light Maintenance,STREET LIGHT OUT,2026-10-30 00:00:00-07:00,Closed,Web,...,005cs00000cYlmbAAC,2025-10-16 22:27:21-07:00,00G61000001Bq9xEAC,5008z000003yjPyAAI,Medium,Problem resolved,Duplicate,0.0,0.0,Closed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1347270,5410043,NaN,2025-10-13 14:21:02-07:00,365.0,Traffic Engineering,Traffic Engineering,TRAFFIC OPERATIONS TRAFFIC CALMING/SAFETY FEA...,2026-10-13 00:00:00-07:00,Closed,Mobile,...,005t0000000F57QAAS,2026-10-13 10:43:05-07:00,005t0000000FFJHAA4,NaN,Low,NaN,Closed,1.0,1.0,Closed
1348572,5431773,NaN,2025-10-30 09:28:00-07:00,365.0,TSW,Street Light Maintenance,STREET LIGHT OUT,2026-10-30 00:00:00-07:00,Closed,Web,...,0058z000000nLRJAA2,2026-06-02 14:13:37-07:00,0058z000000fRmfAAE,NaN,Medium,NaN,Acknowledged,0.0,0.0,Closed
1365949,5075612,5049230.0,2025-01-16 04:28:00-08:00,365.0,TSW,Street Light Maintenance,STREET LIGHT OUT,2026-01-16 00:00:00-08:00,Referred,Web,...,005t0000000F9D3AAK,2026-01-16 09:55:51-08:00,00Gcs000003HyZ7EAK,500cs00000O2qazAAB,Medium,Externally Referred,Closed - Referred,0.0,0.0,Closed - Referred
603301,4508097,4491550.0,2023-11-16 11:00:00-08:00,365.0,TSW ROW,Graffiti - Code Enforcement,Commercial Property,2024-11-15 00:00:00-08:00,Closed,Mobile,...,005t0000001kdcLAAQ,2024-11-15 12:00:10-08:00,00G8z000000XBhmEAG,5008z000006bcPvAAI,Medium,Compliance Achieved,Closed,0.0,0.0,Closed


In [46]:
outliers['case_age_days'].describe()

count    21681.000000
mean       605.699137
std        208.840536
min        365.000000
25%        439.000000
50%        546.000000
75%        715.000000
max       1400.000000
Name: case_age_days, dtype: float64

In [47]:
calculated_age = (outliers['date_closed'] - outliers['date_requested']).dt.days

In [48]:
outliers[(outliers['case_age_days'] - calculated_age).abs() > 1]

,service_request_id,service_request_parent_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,case_origin,...,last_modified_by_id,last_modified_date,owner_id,parent_case_id,priority,case_reason,private_status,internal_case_referral_flag,internal_referral_count,referral_status


In [49]:
outliers[['date_requested', 'date_closed']].describe()

,date_requested,date_closed
count,21681,21681
mean,2023-12-27 07:51:34.875928320-08:00,2025-08-23 19:27:30.809464576-07:00
min,2023-01-01 00:07:00-08:00,2024-01-03 00:00:00-08:00
25%,2023-05-25 03:12:00-07:00,2024-12-16 00:00:00-08:00
50%,2023-10-18 04:45:00-07:00,2025-09-15 00:00:00-07:00
75%,2024-07-15 11:55:00-07:00,2026-05-05 00:00:00-07:00
max,2025-11-06 13:39:28-08:00,2026-11-07 00:00:00-08:00


In [50]:
outliers['case_record_type'].value_counts()

case_record_type
TSW                             12443
TSW ROW                          5277
ESD Complaint/Report             1390
DSD                              1028
Parks & Recreation                611
Traffic Engineering               429
Parking                           239
Storm Water Code Enforcement      181
Neighborhood Policing              78
ESD Abatement                       5
Name: count, dtype: int64

In [51]:
outliers['service_name'].value_counts()

service_name
Street Light Maintenance                   4065
Vegetation Encroachment                    2904
Traffic Sign Maintenance                   2533
Graffiti - Code Enforcement                2076
ROW Maintenance                            2055
Sidewalk Repair Issue                      1645
Tree Maintenance                            641
Stormwater                                  545
Parks Issue                                 522
Traffic Engineering                         522
Environmental Services Code Compliance      456
STRO Violation                              417
Pavement Maintenance                        391
Other                                       357
Illegal Dumping                             355
Encampment                                  302
Graffiti - Private                          248
Missed Collection                           234
Parking Violation                           224
Development Services - Code Enforcement     194
Trash/Recycling Collection 

In [52]:
outliers['service_name_detail'].value_counts()

service_name_detail
STREET LIGHT OUT                   3581
CONCRETE SIDEWALK REPAIR ISSUE     1600
Commercial Property                1217
PAINT STRIPING - MAINTENANCE        754
Other                               585
                                   ... 
DRAIN OUTFALL                         1
RESTORE CONCRETE UTILITY TRENCH       1
Broken Park Fountain                  1
LIGHT UPGRADE                         1
Restroom - Sink                       1
Name: count, Length: 181, dtype: int64

### `internal_referral_count`, `internal_case_referral_flag`

In [53]:
internal_data = df[['internal_referral_count', 'internal_case_referral_flag']]

In [54]:
internal_data.loc[internal_data['internal_case_referral_flag'] == 0, 'internal_referral_count'].unique()

array([0., 1., 2., 6., 5.])

In [55]:
df.loc[df['internal_referral_count'] > 0, 'internal_case_referral_flag'] = 1.0

In [56]:
internal_data.loc[internal_data['internal_referral_count'] >= 1., 'internal_case_referral_flag'].unique()

array([1., 0.])

In [57]:
df.loc[(df['internal_case_referral_flag'] == 1.) & (df['internal_referral_count'] == 0.), ['service_request_id', 'date_requested', 'case_age_days', 'case_record_type', 'service_name', 'date_closed', 'status', 'referred', 'case_reason', 'private_status', 'internal_referral_count', 'internal_case_referral_flag', 'referral_status']]

,service_request_id,date_requested,case_age_days,case_record_type,service_name,date_closed,status,referred,case_reason,private_status,internal_referral_count,internal_case_referral_flag,referral_status
650640,4780714,2024-06-03 15:19:00-07:00,1.0,TSW,Graffiti - Public,2024-06-04 00:00:00-07:00,Referred,caltrans,Problem resolved,Closed - Referred,0.0,1.0,Closed - Referred


In [58]:
df.loc[df['service_request_id'] == 4780714, 'internal_case_referral_flag'] = 0.

### checking timeline

In [59]:
dates = df[['service_request_id', 'date_requested', 'last_modified_date', 'date_closed']]

In [60]:
(dates['date_requested'].dt.normalize() <= dates['last_modified_date'].dt.normalize()).mean()

1.0

In [61]:
df.loc[~(dates['date_requested'].dt.normalize() <= dates['last_modified_date'].dt.normalize()), dates.columns]

,service_request_id,date_requested,last_modified_date,date_closed
